<a href="https://colab.research.google.com/github/lucmos2002/workshopGITHUB/blob/master/06x_pretraining_Lucas_Souza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pré-Treinamento

## 1. Entropia antes e depois do pré-treino
Na aula passada você aprendeu a preparar os dados e fazer o pré-treino do modelo usando dados não rotulados. Vamos colocar isto em prática com novos dados e vamos medir o impacto do pré-treinamento. Compute a entropia do capítulo 6 do livro _The Time Machine_ com o modelo GPT sem nenhum pré-treinamento. Depois faça o pré-treino do modelo usando os cinco primeiros capítulos do mesmo livro e em seguida volte a computar a entropia do capítulo 6. Para isso, faça o seguinte:

- Recrie as classes para o modelo GPT e as funções do data loader.
- Instancie o GPTModel com 124M de parâmetros.
- Passe o texto do capítulo 6 pelo modelo e compute a entropia.
- Faça o pré-treino do modelo usando os 5 primeiros capítulos.
- Passe novamente o texto do capítulo 6 pelo modelo pré-treinado e compute a entropia e veja a diferença.

Execute a célula logo abaixo para fazer o download dos arquivos. Um arquivo contém o texto dos 5 primeiros capítulos do livro que você usará para o pré-treinamento e o outro arquivo possui o texto do capítulo 6 que você usará para computar a entropia. O conteúdo dos arquivos é armazenado nas variáveis `time_machine_ch01_ch05` e `time_machine_ch06`.

#### Download dos arquivos

In [3]:
import requests
import re

def download_file_from_google_drive(file_url, destination):
    file_id = extract_google_drive_id(file_url)
    URL = "https://docs.google.com/uc?export=download&confirm=1"

    session = requests.Session()

    response = session.get(URL, params={"id": file_id}, stream=True)
    token = get_confirm_token(response)

    if token:
        params = {"id": file_id, "confirm": token}
        response = session.get(URL, params=params, stream=True)

    save_response_content(response, destination)


def extract_google_drive_id(url):
    url_pattern = r"/d/([a-zA-Z0-9_-]+)/view\?"
    response = re.search(url_pattern, url)
    if response:
        return response.group(1)
    return None


def get_confirm_token(response):
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            return value

    return None


def save_response_content(response, destination):
    CHUNK_SIZE = 32768

    with open(destination, "wb") as f:
        for chunk in response.iter_content(CHUNK_SIZE):
            if chunk:  # filter out keep-alive new chunks
                f.write(chunk)


# download dos arquivos
url_time_machine_ch01_ch05 = "https://drive.google.com/file/d/1nFeGmveao-WxYt5eAGjTnln2JUu4ugCr/view?usp=sharing"
download_file_from_google_drive(url_time_machine_ch01_ch05, "The Time Machine - ch01 to ch05.txt")

url_time_machine_ch06 = "https://drive.google.com/file/d/16q2ftm6B9wRBO4hhUEUSH8uU-6RgKdxy/view?usp=sharing"
download_file_from_google_drive(url_time_machine_ch06, "The Time Machine - ch06.txt")

# carrega o conteúdo dos arquivos nas variáveis
with open("The Time Machine - ch01 to ch05.txt", "r", encoding="utf-8") as f:
    time_machine_ch01_ch05 = f.read()

with open("The Time Machine - ch06.txt", "r", encoding="utf-8") as f:
    time_machine_ch06 = f.read()

#### Imports
Os imports necessários já estão todos aqui

In [4]:
import os
import urllib
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

#### Dataset e dataLoader

In [5]:
# RECRIE AS CLASSES DO DATASET E DATALOADER
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


#### GPTModel

In [6]:
# RECRIE AS CLASSES PARA CRIAÇÃO DO MODELO GPT

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)   # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (B, T) array of indices in the current context
    for _ in range(max_new_tokens):

        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        # (batch, n_token, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]

        # Get the idx of the vocab entry with the highest logits value
        idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

#### Funções de treinamento e avaliação

In [7]:
# RECRIE AS FUNÇÕES PARA O TREINAMENTO DO MODELO, VISTAS NA ÚLTIMA AULA
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Reduce the number of batches to match the total number of batches in the data loader
        # if num_batches exceeds the number of batches in the data loader
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # add batch dimension
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # remove batch dimension
    return tokenizer.decode(flat.tolist())

# train_model_simple E FUNÇÕES RELACIONADAS

def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    # Initialize lists to track losses and tokens seen
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    # Main training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # Reset loss gradients from previous batch iteration
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward() # Calculate loss gradients
            optimizer.step() # Update model weights using loss gradients
            tokens_seen += input_batch.numel()
            global_step += 1

            # Optional evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Print a sample text after each epoch
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen


def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))  # Compact print format
    model.train()

#### Instanciação do GPT 124M e tokenizer

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

# INSTANCIE O GPT PASSANDO A CONFIGURAÇÃO ACIMA
# INSTANCIE TAMBÉM O TOKENIZER TIKTOKEN USANDO O GPT2 ENCONDING

model = GPTModel(GPT_CONFIG_124M).to(device)
tokenizer = tiktoken.get_encoding("gpt2")

#### Entropia com o modelo sem nenhum pré-treino

In [9]:
# CARREGE O CAPÍTULO 6 NO DATA LOADER E COMPUTE A ENTROPIA (A LOSS USADA NO TREINO)
# PASSANDO O TEXTO PELO MODELO SEM TREINAMENTO

time_machine_ch06 = time_machine_ch06.replace("\n", " ")
data = create_dataloader_v1(time_machine_ch06, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0)
loss = calc_loss_loader(data, model, device)
print(f"Entropia do capítulo 6: {loss:.3f}")

Entropia do capítulo 6: 11.003


#### Pré-treino com os capítulos 1 a 5

In [10]:
# Dados de treino e validção
train_ratio = 0.90
split_idx = int(train_ratio * len(time_machine_ch01_ch05))
train_data = time_machine_ch01_ch05[:split_idx]
val_data = time_machine_ch01_ch05[split_idx:]

# parâmetros de treinamento
num_epochs = 10
eval_freq = 5
eval_iter = 5
start_context = "The Time Machine."
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)


# CRIE OS DATA LOADERS PARA OS DADOS DE TREINO E VALIDAÇÃO DEFINIDOS ACIMA
# EM SEGUIDA, CHAME A FUNÇÃO DE TREINO PASSANDO OS PARÂMETROS DEFINIDOS ACIMA

torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

# Chamada do Treinamento
train_losses, val_losses, tokens_seen = train_model_simple(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs,
    eval_freq=eval_freq,
    eval_iter=eval_iter,
    start_context=start_context,
    tokenizer=tokenizer,
)




Ep 1 (Step 000000): Train loss 9.877, Val loss 10.015
Ep 1 (Step 000005): Train loss 8.009, Val loss 8.504
Ep 1 (Step 000010): Train loss 6.798, Val loss 7.649
Ep 1 (Step 000015): Train loss 6.302, Val loss 7.425
Ep 1 (Step 000020): Train loss 5.836, Val loss 7.219
The Time Machine. I, and of the of the of the of the of the of the, and, and, and of the, and, and, and the of the of the of the of the of the of the of the of the of the of the
Ep 2 (Step 000025): Train loss 5.392, Val loss 7.328
Ep 2 (Step 000030): Train loss 5.435, Val loss 7.368
Ep 2 (Step 000035): Train loss 5.126, Val loss 7.386
Ep 2 (Step 000040): Train loss 5.044, Val loss 7.256
The Time Machine. ” ” ” ” ” ” said the Time Traveller. ” ” ” ” ” said the Time Traveller. ” �
Ep 3 (Step 000045): Train loss 5.096, Val loss 7.302
Ep 3 (Step 000050): Train loss 4.566, Val loss 7.188
Ep 3 (Step 000055): Train loss 4.416, Val loss 7.298
Ep 3 (Step 000060): Train loss 4.248, Val loss 7.205
The Time Machine. “I” “I“I” “It“It“I” 

#### Entropia do capítulo 6 após o pré-treino com os capítulos 1 a 5

In [11]:
# PASSE NOVAMENTE O CAPÍTULO 6 PELO MODELO, AGORA PRÉ-TREINADO E COMPUTE A
# ENTROPIA NOVAMENTE E VEJA A DIFERENÇA
time_machine_ch06 = time_machine_ch06.replace("\n", " ")
data = create_dataloader_v1(time_machine_ch06, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0)
loss = calc_loss_loader(data, model, device)
print(f"Entropia do capítulo 6: {loss:.3f}")

Entropia do capítulo 6: 8.123


## 2. Entropia com pesos do Hugging Face
Crie uma nova instância do modelo GPT a partir dos pesos pré-treinados no Hugging Face e calcule novamente a entropia do texto do capítulo 6

In [12]:
from types import new_class
file_name = "gpt2-small-124M.pth"
url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

if not os.path.exists(file_name):
    urllib.request.urlretrieve(url, file_name)
    print(f"Downloaded to {file_name}")

BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True,        # Query-key-value bias
    "emb_dim": 768,
    "n_layers": 12,
    "n_heads": 12
}

# INSTANCIE O MODELO GPT COM A BASE_CONFIG DEFINIDA ACIMA
# (USE A MESMA CLASSE QUE VC DEFINIU NO INÍCIO DESTE NOTEBOOK)
# E CARREGUE OS PESOS A PARTIR DO ARQUIVO BAIXADO PELO COMANDO ACIMA

new_model = GPTModel(BASE_CONFIG)
new_model.load_state_dict(torch.load(file_name))
new_model.eval()
new_model.to(device)


Downloaded to gpt2-small-124M.pth


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [14]:
# CRIE O DATA LOADER COM O TEXTO DO CAPÍTULO 6, PASSE PELO MODELO E COMPUTE A
# ENTROPIA

new_data = create_dataloader_v1(time_machine_ch06, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True, num_workers=0)
new_loss = calc_loss_loader(new_data, new_model, device)
print(f"Entropia do capítulo 6: {new_loss:.3f}")

Entropia do capítulo 6: 3.978
